# Lightweight graph prototype

This notebook focuses on the graph idea: photos become nodes, and edges are created from simple, maintainable signals.

The objective is not to replace vector search. The graph is a product layer for grouping, recommendations and navigation.

## Edge signals

Suggested production signals:

- same shooting and close timestamp,
- same detected person,
- visual nearest neighbors from Qdrant,
- same selected tags,
- user feedback such as favorites or client selections.

These signals are easier to debug than opaque text clusters.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'ml' / 'experiments'))

from photo_strategy_benchmark import generate_photos, hybrid_similarity_graph

photos = generate_photos(count=800, seed=7)
hybrid_similarity_graph(photos)

In [ ]:
for top_k in [2, 4, 8]:
    result = hybrid_similarity_graph(photos, top_k=top_k)
    print('top_k=', top_k, result)

## Product interpretation

A small `top_k` avoids one giant component and creates focused groups. A bigger `top_k` is better for exploration but can over-connect the graph.

For a first production version, prefer conservative edges and expose graph groups as suggestions, not as irreversible labels.

## Implementation shape

Tables or collections:

- `photos`: id, library_id, shooting_id, storage_key, metadata,
- `photo_vectors`: photo_id, vector_id or Qdrant point id,
- `photo_edges`: source_photo_id, target_photo_id, edge_type, weight,
- `photo_groups`: optional cached groups for UI.

Jobs:

- `embed_photo_batch`,
- `refresh_similarity_edges`,
- `refresh_groups_for_shooting`,
- optional `caption_selected_album`.